# Lesson 01: Reading FIX messages
**Level:** beginner · **You need:** nothing. Read [Lesson 00](00_market_structure_primer.md) first if you can.

**Goal of this lesson:** look at a line of FIX and understand it the way you understand a sentence. By the end you
will be able to take any message from the log files, say what it is, who sent it, what it asks for, and follow
one client order from the bridge to the LP and back.

> **How to read these notebooks**
> * The grey boxes are Python code. **You don't need to understand the code.** Above every code box there is a
>   line starting with 🔧 that says in plain words what the code does. Read that, then read the **output** below
>   the box. The output is what matters.
> * Every time a FIX tag appears, its name and meaning are repeated next to it, e.g. `35=A` (MsgType = **Logon**).
>   You never need to remember anything from a previous section.
> * At the end there is a **cheat sheet** and then **10 worked examples**: real messages from the data, explained
>   field by field. You can jump there any time.

### Words used in this lesson
| Word | Meaning |
|---|---|
| **FIX** | *Financial Information eXchange*. The standard "language" that brokers, banks and LPs use to send each other prices, orders and fills. It is plain text |
| **LP** | *Liquidity provider*. A bank or market maker that gives the broker prices and fills its orders. Here: LP_A, LP_B, LP_C |
| **Bridge** | The broker's software that connects to the LPs over FIX. Here it is called `BRIDGE01` |
| **Session** | One open FIX connection between the bridge and one LP (like an open phone call). We have 3 sessions: bridge↔LP_A, bridge↔LP_B, bridge↔LP_C |
| **Tag** | The number before the `=` in `35=A`. Each number has a fixed meaning (35 always means "message type") |
| **Log file** | A text file where the bridge writes down every message it sends or receives, one per line |

### The data
`data/fix_logs/BRIDGE_LP_A.log`, `BRIDGE_LP_B.log`, `BRIDGE_LP_C.log`: one file per LP session. Each line looks like:

```
20251015-11:29:58.500 OUT 8=FIX.4.4|9=73|35=A|49=BRIDGE01|56=LP_B|...|10=117|
└── when the bridge ──┘ └┬┘ └──────────────── the FIX message itself ────────────────┘
    wrote this line      │
                         OUT = our bridge SENT it to the LP · IN = we RECEIVED it from the LP
```

Inside a real FIX message the fields are separated by an invisible character called **SOH** (character number 1
in the ASCII table). Screens can't show it, so log viewers, and this lab, display it as `|`.

🔧 **The code below** opens the LP_B log file and prints its first 8 lines, so you can see what raw FIX looks like.

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
from bridgelab import fix
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 120)

with open("../data/fix_logs/BRIDGE_LP_B.log") as f:
    lines = [next(f) for _ in range(8)]
for line in lines:
    print(line.rstrip()[:200])

20251015-11:29:58.500 OUT 8=FIX.4.4|9=73|35=A|49=BRIDGE01|56=LP_B|34=1|52=20251015-11:29:58.500|98=0|108=10|141=Y|10=117|
20251015-11:29:58.518 IN  8=FIX.4.4|9=73|35=A|49=LP_B|56=BRIDGE01|34=1|52=20251015-11:29:58.509|98=0|108=10|141=Y|10=126|
20251015-11:29:58.521 OUT 8=FIX.4.4|9=121|35=V|49=BRIDGE01|56=LP_B|34=2|52=20251015-11:29:58.521|262=MD-LP_B|263=1|264=2|265=0|146=3|55=EURUSD|55=GBPUSD|55=XAUUSD|10=227|
20251015-11:30:00.457 IN  8=FIX.4.4|9=218|35=W|49=LP_B|56=BRIDGE01|34=2|52=20251015-11:30:00.449|262=MD-LP_B-XAUUSD|55=XAUUSD|268=4|269=0|270=4085.33|271=200|290=1|269=0|270=4085.30|271=400|290=2|269=
20251015-11:30:00.644 IN  8=FIX.4.4|9=218|35=W|49=LP_B|56=BRIDGE01|34=3|52=20251015-11:30:00.634|262=MD-LP_B-XAUUSD|55=XAUUSD|268=4|269=0|270=4085.36|271=300|290=1|269=0|270=4085.33|271=600|290=2|269=
20251015-11:30:01.210 IN  8=FIX.4.4|9=234|35=W|49=LP_B|56=BRIDGE01|34=4|52=20251015-11:30:01.201|262=MD-LP_B-GBPUSD|55=GBPUSD|268=4|269=0|270=1.33475|271=2000000|290=1|269=0|270=1.334

**What you are looking at:** the very start of the trading day on the LP_B connection.
* Line 1 `OUT ... 35=A`: our bridge sends a **Logon** (`35` = MsgType, `A` = Logon) to LP_B: "hello, I want to
  connect".
* Line 2 `IN ... 35=A`: LP_B answers with its own **Logon**: "hello, connection accepted".
* Then `35=V` (MsgType = **MarketDataRequest**): the bridge asks LP_B "please stream me prices for EURUSD,
  GBPUSD and XAUUSD".
* Then many `35=W` (MsgType = **MarketDataSnapshot**): LP_B starts sending prices.

Don't worry if the lines still look like noise. The next sections take one line apart, piece by piece.

---
## 1. Split a message into tag=value fields

A FIX message is just a list of pairs written as `number=value`, separated by `|` (the SOH character).
The number is the **tag** and tells you what the value means. So to read a message you:
1. cut it at every `|`
2. cut each piece at the first `=`: left side = tag, right side = value
3. look up what each tag number means (FIX has a public dictionary of tags)

🔧 **The code below** takes line 1 (the bridge's Logon to LP_B), cuts it into pieces and shows each tag, its value
and the tag's official name in a table.

In [2]:
log_time, direction, raw = lines[0][:21], lines[0][22:25].strip(), lines[0][26:].strip()
print("logged at:", log_time, "| direction:", direction)
print(raw)

fields = [part.split("=", 1) for part in raw.split("|") if part]
pd.DataFrame(fields, columns=["tag", "value"]).assign(name=lambda d: d.tag.astype(int).map(fix.TAGS))

logged at: 20251015-11:29:58.500 | direction: OUT
8=FIX.4.4|9=73|35=A|49=BRIDGE01|56=LP_B|34=1|52=20251015-11:29:58.500|98=0|108=10|141=Y|10=117|


,tag,value,name
0,8,FIX.4.4,BeginString
1,9,73,BodyLength
2,35,A,MsgType
3,49,BRIDGE01,SenderCompID
4,56,LP_B,TargetCompID
5,34,1,MsgSeqNum
6,52,20251015-11:29:58.500,SendingTime
7,98,0,EncryptMethod
8,108,10,HeartBtInt
9,141,Y,ResetSeqNumFlag


**Reading the table row by row, in plain English:**

| Field | Tag name | What it says |
|---|---|---|
| `8=FIX.4.4` | BeginString | "This message uses FIX version 4.4". Always the first field |
| `9=73` | BodyLength | "The body of this message is 73 characters long". Used to check nothing was lost (section 3) |
| `35=A` | MsgType | The **type** of message. `A` = **Logon**. This is the most important field: it tells you how to read the rest |
| `49=BRIDGE01` | SenderCompID | **Who sent it**: our bridge |
| `56=LP_B` | TargetCompID | **Who it is for**: LP_B |
| `34=1` | MsgSeqNum | **Message number**. This is message number 1 that the bridge sends to LP_B today. Each next message gets +1 (2, 3, 4…). Lesson 02 is all about these numbers |
| `52=20251015-11:29:58.500` | SendingTime | **When it was sent**: 15 Oct 2025, 11:29:58 and 500 milliseconds, in UTC time, on the sender's clock |
| `98=0` | EncryptMethod | `0` = no encryption inside the message (normal; the network line itself is secured) |
| `108=10` | HeartBtInt | Heartbeat interval: "if we have nothing to say, we'll each send a small 'I'm alive' message every **10 seconds**" |
| `141=Y` | ResetSeqNumFlag | "Let's both **restart message numbering from 1**". Normal at the start of the day |
| `10=117` | CheckSum | A control number calculated from all the characters before it, to detect corruption (section 3). Always the last field |

**The whole message in one sentence:** *"Bridge to LP_B, message #1, sent at 11:29:58.500: I want to log on;
let's heartbeat every 10 seconds and both start counting messages from 1."*

---
## 2. Header, body and trailer

Every FIX message has the same 3 parts, like a letter:

| Part | Like a letter's… | Tags | Rule |
|---|---|---|---|
| **Header** | envelope: from, to, what kind of letter, letter number, date | `8` BeginString (version), `9` BodyLength (size), `35` MsgType (what kind of message), `49` SenderCompID (from), `56` TargetCompID (to), `34` MsgSeqNum (message number), `52` SendingTime (when). Sometimes also `43` PossDupFlag and `122` OrigSendingTime (used when a message is sent a second time, Lesson 02) | `8`, `9` and `35` must always be the first three fields, in this order |
| **Body** | the letter itself | depends on the message type: an order has symbol, side, quantity, price; a logon has heartbeat interval… | |
| **Trailer** | the seal | `10` CheckSum | always the very last field |

In the Logon above: the header is `8, 9, 35, 49, 56, 34, 52`, the body is `98=0|108=10|141=Y|`, and the trailer
is `10=117|`.

---
## 3. Validate BodyLength (9) and CheckSum (10) by hand

Networks and software sometimes damage data: a character gets lost, changed or added. FIX protects every message
with two control numbers that the receiver recalculates. If either doesn't match, the receiver throws the message
away.

### BodyLength, tag `9`: count the characters
**Rule:** count every character that comes **after** the `9=73|` field and **before** `10=`. Each `|` separator
counts as 1 character.

For our Logon `8=FIX.4.4|9=73|` **`35=A|49=BRIDGE01|56=LP_B|34=1|52=20251015-11:29:58.500|98=0|108=10|141=Y|`** `10=117|`
(the bold part is what we count):

| Piece (with its `\|`) | Characters |
|---|---|
| `35=A\|` | 5 |
| `49=BRIDGE01\|` | 12 |
| `56=LP_B\|` | 8 |
| `34=1\|` | 5 |
| `52=20251015-11:29:58.500\|` | 25 |
| `98=0\|` | 5 |
| `108=10\|` | 7 |
| `141=Y\|` | 6 |
| **Total** | **73** ✔ the same as `9=73`, so nothing was lost or added |

### CheckSum, tag `10`: add up character codes
**Rule:** every character has a number in the ASCII table (for example `8` = 56, `=` = 61, `F` = 70, `A` = 65,
and the SOH separator = 1). Add up the numbers of **every character before `10=`** (including `8=FIX.4.4|9=73|`),
divide by 256 and keep the **remainder**. Write it with 3 digits.

* Example of the first characters: `8` → 56, `=` → 61, `F` → 70, `I` → 73, `X` → 88 …
* For this Logon, all characters before `10=` add up to **4,469**.
* 4,469 ÷ 256 = 17, remainder **117** (because 17 × 256 = 4,352 and 4,469 − 4,352 = 117).
* So the checksum must be `10=117` ✔.

⚠️ The checksum is calculated with the **real SOH character (value 1)**, not with the `|` you see on screen (`|`
has value 124). That's why the code below first swaps every `|` back to SOH.

🔧 **The code below** does exactly the two calculations above on the Logon message and prints the number written in
the message ("declared") next to the number we calculated ("actual").

In [3]:
SOH = "\x01"
msg = raw.replace("|", SOH)
start_body = msg.index(SOH, msg.index("9=")) + 1           # character after "9=73<SOH>"
start_trailer = msg.rindex(SOH + "10=") + 1                 # position of "10="
body = msg[start_body:start_trailer]
print("BodyLength declared:", msg.split(SOH)[1].split("=")[1], "| actual:", len(body))
print("CheckSum declared:  ", msg[start_trailer + 3:start_trailer + 6],
      "| actual:", f"{sum(msg[:start_trailer].encode()) % 256:03d}")

BodyLength declared: 73 | actual: 73
CheckSum declared:   117 | actual: 117


Both match, so the message arrived intact.

**What if one character changes?** Below we take a real order the bridge sent to LP_B and secretly change its
price: we put a `9` in front of the value of tag `44` (Price), so `44=4085.67` becomes `44=94085.67`.

🔧 **The code below** prints the real order, checks it, then checks the tampered copy. `fix.validate()` is a helper
in this repo that does the same two calculations you just saw.

In [4]:
order_line = next(l for l in open("../data/fix_logs/BRIDGE_LP_B.log") if "35=D|" in l)
order_raw = order_line[26:].strip()
print(order_raw)
print("original :", fix.validate(order_raw))
tampered = order_raw.replace("|44=", "|44=9")               # someone changes the price
print("tampered :", fix.validate(tampered))

8=FIX.4.4|9=148|35=D|49=BRIDGE01|56=LP_B|34=6|52=20251015-11:30:34.547|11=BR700004-3|1=BRIDGE01|55=XAUUSD|54=2|60=20251015-11:30:34.547|38=490|40=2|44=4085.67|59=3|10=218|
original : {'body_length_declared': 148, 'body_length_actual': 148, 'checksum_declared': '218', 'checksum_actual': '218', 'body_length_ok': True, 'checksum_ok': True, 'valid': True}
tampered : {'body_length_declared': 148, 'body_length_actual': 149, 'checksum_declared': '218', 'checksum_actual': '019', 'body_length_ok': False, 'checksum_ok': False, 'valid': False}


**How to read the output:**
* Original: BodyLength declared 148 = actual 148, CheckSum declared 218 = actual 218 → `valid: True`.
* Tampered: we added one character, so the actual BodyLength becomes **149** (but the message still says 148),
  and the actual CheckSum becomes **019** (but the message still says 218) → `valid: False`.

**What happens in real life:** the LP's FIX software would refuse this message and send back a **Reject**
(`35=3`, MsgType = session-level Reject) instead of executing the order. In practice you almost never calculate
these by hand, because the FIX software does it automatically. You need to understand them because errors like
*"Invalid checksum"* or *"Incorrect BodyLength"* show up when a new LP connection is being tested, and you
need to know what they mean.

---
## 4. Decode the message: tag names and enum values

Many values in FIX are **codes**: `54=2` doesn't say "sell", it says "2", and you have to know that for tag 54
(Side) "2" means Sell. A list of allowed codes is called an **enum** (enumeration).

🔧 **The code below** runs `fix.explain()` on the order from section 3. It puts, next to each field, the tag's
name and the meaning of the code.

In [5]:
pd.DataFrame(fix.explain(order_raw))

,tag,name,value,meaning
0,8,BeginString,FIX.4.4,
1,9,BodyLength,148,
2,35,MsgType,D,NewOrderSingle
3,49,SenderCompID,BRIDGE01,
4,56,TargetCompID,LP_B,
5,34,MsgSeqNum,6,
6,52,SendingTime,20251015-11:30:34.547,
7,11,ClOrdID,BR700004-3,
8,1,Account,BRIDGE01,
9,55,Symbol,XAUUSD,


**The order in plain English, field by field:**

| Field | Tag name | Meaning |
|---|---|---|
| `35=D` | MsgType | `D` = **NewOrderSingle**: "here is a new order" |
| `49=BRIDGE01` → `56=LP_B` | Sender → Target | from our bridge to LP_B |
| `34=6` | MsgSeqNum | this is the bridge's 6th message to LP_B today |
| `11=BR700004-3` | ClOrdID (Client Order ID) | **the bridge's own reference for this order**. Read it as: `BR` + client order number `700004` (from the trading platform) + `-3` = **3rd attempt** (the first two attempts went to other LPs) |
| `1=BRIDGE01` | Account | the account at the LP |
| `55=XAUUSD` | Symbol | gold priced in US dollars |
| `54=2` | Side | `1` = Buy, `2` = **Sell** |
| `60=…` | TransactTime | when the order was created |
| `38=490` | OrderQty | quantity: **490** (for gold at this LP: ounces) |
| `40=2` | OrdType (order type) | `1` = Market (any price), `2` = **Limit** (only at this price or better) |
| `44=4085.67` | Price | the limit price: sell at **4085.67 or higher** |
| `59=3` | TimeInForce (how long the order lives) | `3` = **IOC = Immediate Or Cancel**: fill what you can *right now*, cancel the rest. `1` would be GTC (stays until cancelled) |

**In one sentence:** *"Bridge to LP_B: sell 490 ounces of gold at 4085.67 or better, right now or not at all;
my reference is BR700004-3."*

Why limit + IOC? The bridge wants the price the LP was showing a moment ago. If the LP can't give that price
right now, the bridge would rather be told "no" immediately and try another LP.

---
## 5. Build a message yourself

Building a message is reading in reverse: you choose the type and the fields, and the BodyLength (`9`) and
CheckSum (`10`) are calculated from them.

🔧 **The code below** builds two small session messages with `fix.build()`:
* a **Heartbeat** (`35=0`): "I'm alive, nothing else to say", message number 1234;
* a **TestRequest** (`35=1`): "are you there? please answer with this ID", with `112=PING-1`
  (TestReqID = the ID the other side must repeat in its answer).

It then checks both with `fix.validate()`.

In [6]:
hb = fix.build("0", "BRIDGE01", "LP_A", 1234, "20251015-12:00:00.000", [], sep="|")
tr = fix.build("1", "BRIDGE01", "LP_A", 1235, "20251015-12:00:05.000", [(112, "PING-1")], sep="|")
print(hb, fix.validate(hb)["valid"])
print(tr, fix.validate(tr)["valid"])

8=FIX.4.4|9=58|35=0|49=BRIDGE01|56=LP_A|34=1234|52=20251015-12:00:00.000|10=159| True
8=FIX.4.4|9=69|35=1|49=BRIDGE01|56=LP_A|34=1235|52=20251015-12:00:05.000|112=PING-1|10=006| True


Both print `True`: the BodyLength and CheckSum were calculated correctly. Notice that a Heartbeat has **no body
at all**: just the header and the trailer. Its whole job is to exist.

In real life, building messages by hand happens during **certification**: before going live with a new LP, both
sides exchange test messages in a test environment ("UAT") to prove they understand each other.

---
## 6. Repeating groups: a market-data snapshot (35=W)

A price update (`35=W`, MsgType = **MarketDataSnapshotFullRefresh**, "here is my full current price list") contains
**several prices** in one message: bids and offers at different levels. FIX handles lists with a **repeating
group**:
* first a **counter**: `268=4` (NoMDEntries = "number of market-data entries" → 4 entries follow)
* then the same set of tags repeated 4 times, once per entry:
  * `269` MDEntryType: `0` = **Bid** (price the LP will **buy** at) or `1` = **Offer** (price the LP will **sell** at, also called *ask*)
  * `270` MDEntryPx: the price
  * `271` MDEntrySize: how much is available at that price
  * `290` MDPriceLevel: `1` = best price, `2` = second best

Because tags like `270` appear 4 times in the same message, you must read them **in order**, entry by entry.

🔧 **The code below** finds the first EURUSD price update from LP_B, prints it raw, and then shows the 4 entries
as a small table.

In [7]:
md_raw = next(l for l in open("../data/fix_logs/BRIDGE_LP_B.log") if "35=W|" in l and "EURUSD" in l)[26:].strip()
print(md_raw)
m = fix.parse(md_raw)
book = pd.DataFrame(m["groups"]).rename(columns={269: "side", 270: "price", 271: "size", 290: "level"})
book["side"] = book.side.map({"0": "BID", "1": "OFFER"})
book

8=FIX.4.4|9=235|35=W|49=LP_B|56=BRIDGE01|34=10|52=20251015-11:30:03.445|262=MD-LP_B-EURUSD|55=EURUSD|268=4|269=0|270=1.16247|271=1500000|290=1|269=0|270=1.16244|271=3000000|290=2|269=1|270=1.16253|271=1500000|290=1|269=1|270=1.16256|271=3000000|290=2|10=003|


,side,price,size,level
0,BID,1.16247,1500000,1
1,BID,1.16244,3000000,2
2,OFFER,1.16253,1500000,1
3,OFFER,1.16256,3000000,2


**Reading the table:** LP_B says, for EURUSD:
* "I will **buy** 1,500,000 EUR at 1.16247 (level 1), and another 3,000,000 at 1.16244 (level 2)."
* "I will **sell** 1,500,000 EUR at 1.16253 (level 1), and another 3,000,000 at 1.16256 (level 2)."

The best bid (1.16247) is always lower than the best offer (1.16253). The difference is the **spread**, which
is how the LP earns money.

🔧 **The code below** takes the level-1 bid and offer and calculates the spread in **pips**
(for EURUSD, 1 pip = 0.0001, the 4th decimal).

In [8]:
best_bid = float(book[(book.side == "BID") & (book.level == "1")].price.iloc[0])
best_ask = float(book[(book.side == "OFFER") & (book.level == "1")].price.iloc[0])
print(f"LP_B EURUSD top of book {best_bid} / {best_ask} -> spread = {(best_ask - best_bid) / 0.0001:.1f} pips")

LP_B EURUSD top of book 1.16247 / 1.16253 -> spread = 0.6 pips


1.16253 − 1.16247 = 0.00006 = **0.6 pips**. "Top of book" = the level-1 (best) bid and offer.

---
## 7. Message types in a whole session log

Now let's step back from one message to the whole day on the LP_B connection.

🔧 **The code below** reads every line of the LP_B log, finds each message's type (tag `35`, MsgType), and counts
how many of each type were sent (OUT) and received (IN).

In [9]:
rows = []
for log_time, direction, raw_msg in fix.read_log("../data/fix_logs/BRIDGE_LP_B.log"):
    f = fix.parse(raw_msg)
    rows.append(dict(log_time=log_time, direction=direction, msg_type=f[35], name=fix.MSG_TYPES.get(f[35]),
                     seq=int(f[34]), raw=raw_msg))
log = pd.DataFrame(rows)
log.groupby(["direction", "msg_type", "name"]).size().rename("messages").reset_index()

,direction,msg_type,name,messages
0,IN,4,SequenceReset,2
1,IN,8,ExecutionReport,168
2,IN,A,Logon,1
3,IN,W,MarketDataSnapshotFullRefresh,22149
4,OUT,0,Heartbeat,657
5,OUT,2,ResendRequest,1
6,OUT,A,Logon,1
7,OUT,D,NewOrderSingle,142
8,OUT,V,MarketDataRequest,1


**How to read it** (`msg_type` is the value of tag `35`, MsgType):
* **IN `W`** (MarketDataSnapshot = prices from LP_B): by far the largest number. A price connection is mostly prices.
* **OUT `D`** (NewOrderSingle = our orders to LP_B) and **IN `8`** (ExecutionReport = LP_B's answers: filled,
  rejected, cancelled): a few hundred.
* **OUT `0`** (Heartbeat = "I'm alive"): a side sends one only after 10 seconds without sending anything else.
  Our bridge mostly listens on this connection (it only sends the occasional order), so it sends hundreds of
  heartbeats. LP_B sends **none**: it is sending prices all the time, and that already proves it's alive.
* **`A`** (Logon) once each way at the start.
* Rare ones such as **`2`** (ResendRequest = "please send me again messages I missed") and **`4`**
  (SequenceReset = "skip these message numbers") mean **something went wrong**. Lesson 02 investigates them.

In real life, this kind of count is the first quick health check of a connection: a type that suddenly appears or
disappears tells you where to look.

---
## 8. Follow one order end to end (D → 8 → re-route → 8)

This is the most common investigation at work: *"A client says his order was slow / rejected / filled at a bad
price. What happened?"*

Remember how orders are linked:
* The bridge sends an order: `35=D` (MsgType = NewOrderSingle).
* The LP answers with one or more `35=8` (MsgType = **ExecutionReport**, "here is what happened to your order").
* Both carry the same `11` (**ClOrdID**, the bridge's reference), written `BR<client order number>-<attempt>`.
  So `BR700335-1` and `BR700335-2` are attempt 1 and attempt 2 of the **same** client order 700335.
* In the answer, `150` (**ExecType**, what just happened) tells you the result: `F` = Trade (filled), `8` =
  Rejected, `4` = Canceled. `58` (**Text**) gives the reason for a reject.

🔧 **The code below** reads all orders and answers from all three LP logs into one table, then picks the first
client order that was **rejected by one LP and sent to another** ("re-routed"), and shows its full story in time
order.

In [10]:
all_rows = []
for lp in ["LP_A", "LP_B", "LP_C"]:
    for log_time, direction, raw_msg in fix.read_log(f"../data/fix_logs/BRIDGE_{lp}.log"):
        if "35=D|" in raw_msg or "35=8|" in raw_msg:
            f = fix.parse(raw_msg)
            all_rows.append(dict(lp=lp, log_time=log_time, direction=direction, msg=fix.MSG_TYPES[f[35]],
                                 cl_ord_id=f.get(11), exec_type=fix.ENUMS[150].get(f.get(150), ""),
                                 side=fix.ENUMS[54][f[54]], qty=f.get(38), price=f.get(44) or f.get(31),
                                 last_qty=f.get(32), text=f.get(58, "")))
orders_fix = pd.DataFrame(all_rows)
orders_fix["client_order"] = orders_fix.cl_ord_id.str.extract(r"BR(\d+)-")[0]
rerouted = orders_fix[(orders_fix.exec_type == "Rejected")].client_order
example = orders_fix[orders_fix.client_order.isin(rerouted)].client_order.iloc[0]
orders_fix[orders_fix.client_order == example].sort_values("log_time").drop(columns="client_order")

,lp,log_time,direction,msg,cl_ord_id,exec_type,side,qty,price,last_qty,text
1157,LP_C,20251015-11:34:24.919,OUT,NewOrderSingle,BR700056-1,,Sell,0.37,4086.48,NaN,
1158,LP_C,20251015-11:34:25.063,IN,ExecutionReport,BR700056-1,Rejected,Sell,0.37,NaN,NaN,Off quotes
23,LP_A,20251015-11:34:25.064,OUT,NewOrderSingle,BR700056-2,,Sell,37,4086.45,NaN,
24,LP_A,20251015-11:34:25.071,IN,ExecutionReport,BR700056-2,Trade,Sell,37,4086.45,37,


**The table columns:** `lp` which connection · `log_time` when the bridge logged it · `direction` OUT = we sent,
IN = we received · `msg` message type · `cl_ord_id` tag 11 ClOrdID (our reference) · `exec_type` tag 150 ExecType
(result) · `side` tag 54 · `qty` tag 38 OrderQty · `price` tag 44 Price (in orders) or tag 31 LastPx (fill price,
in answers) · `last_qty` tag 32 LastQty (quantity filled in this answer) · `text` tag 58 (reason).

**The story of client order 700056** (client C1031 wants to **sell 0.37 lots of gold**; 1 lot of gold = 100
ounces, so 37 oz):

| Time | What happened |
|---|---|
| 11:34:24.919 | **OUT `D`** (NewOrderSingle) to LP_C: attempt `BR700056-1`, sell at limit 4086.48, IOC (fill now or cancel) |
| 11:34:25.063 | **IN `8`** (ExecutionReport) from LP_C: ExecType **Rejected**, Text *"Off quotes"* = "that price is no longer available". LP_C took **144 ms** to say no |
| 11:34:25.064 | **OUT `D`** to LP_A: 1 ms later the bridge **re-routes** the same order as attempt `BR700056-2`, at LP_A's price 4086.45 |
| 11:34:25.071 | **IN `8`** from LP_A: ExecType **Trade**, all 37 filled at 4086.45, answered in 7 ms |

**What this tells you:**
* The client was filled, but at **4086.45 instead of 4086.48**: 0.03 USD per ounce worse, because the first LP
  refused. On the platform the client asked for 4086.28 and got 4086.25 (both are the LP price minus the broker's
  markup). That difference is called **slippage**: Lesson 06 measures it.
* LP_C held the order 144 ms before refusing; LP_A answered in 7 ms. An LP that holds orders and then refuses is
  using **last look** ("let me check the price again before I accept"). Lesson 05 tests whether LP_C uses it fairly.
* **Look at the quantity column again:** LP_C was sent **0.37**, LP_A was sent **37**. Same order, 100 times
  different. LP_A receives gold in ounces, LP_C is receiving the number of *lots*. That's a setup mistake in the
  bridge (incident S10). Keep it in mind; section 9 shows it again.

The time between an order (`D`) and its answer (`8`) is the LP's **round trip**: network time plus the time the LP
takes to decide. Lesson 05 measures it for every order.

**In real life** you would do the same thing in the bridge's order screen or by searching the log for
`BR700056`, then write the timeline in the support ticket.

---
## 9. Partial fills

Sometimes an LP doesn't have enough at that price. Then it fills **part** of the order and, because the order is
IOC (Immediate Or Cancel, tag `59=3`), cancels the rest. You see **two** ExecutionReports (`35=8`) for one
order:
1. `150=F` (ExecType = Trade) with `39=1` (OrdStatus = **Partially filled**), `32` (LastQty) = how much was filled
   and `151` (LeavesQty) = how much is still open;
2. `150=4` (ExecType = **Canceled**) with `58=IOC remainder cancelled` (Text = the reason).

The bridge then sends what's left to the next LP.

🔧 **The code below** finds the first order that had a cancelled remainder and shows all its messages.

In [11]:
partial = orders_fix[orders_fix.text.str.contains("IOC remainder")].client_order.iloc[0]
orders_fix[orders_fix.client_order == partial].sort_values("log_time").drop(columns="client_order")

,lp,log_time,direction,msg,cl_ord_id,exec_type,side,qty,price,last_qty,text
1133,LP_C,20251015-11:30:34.405,OUT,NewOrderSingle,BR700004-1,,Sell,9.4,4085.76,NaN,
1134,LP_C,20251015-11:30:34.536,IN,ExecutionReport,BR700004-1,Trade,Sell,9.4,4085.76,1.5,
0,LP_A,20251015-11:30:34.537,OUT,NewOrderSingle,BR700004-2,,Sell,790,4085.73,NaN,
1135,LP_C,20251015-11:30:34.537,IN,ExecutionReport,BR700004-1,Canceled,Sell,9.4,NaN,NaN,IOC remainder cancelled
1,LP_A,20251015-11:30:34.546,IN,ExecutionReport,BR700004-2,Trade,Sell,790,4085.73,300,
2,LP_A,20251015-11:30:34.547,IN,ExecutionReport,BR700004-2,Canceled,Sell,790,NaN,NaN,IOC remainder cancelled
823,LP_B,20251015-11:30:34.547,OUT,NewOrderSingle,BR700004-3,,Sell,490,4085.67,NaN,
824,LP_B,20251015-11:30:34.585,IN,ExecutionReport,BR700004-3,Trade,Sell,490,4085.67,200,
825,LP_B,20251015-11:30:34.586,IN,ExecutionReport,BR700004-3,Canceled,Sell,490,NaN,NaN,IOC remainder cancelled


**The story of client order 700004** (client wants to **sell 9.4 lots of gold**; 1 lot of gold = 100 ounces, so
940 oz):

| Attempt (tag 11, ClOrdID) | Sent to | Quantity sent (tag 38) | Filled (tag 32, LastQty) | Rest |
|---|---|---|---|---|
| `BR700004-1` | LP_C | **9.4** | 1.5 | cancelled (`150=4`) |
| `BR700004-2` | LP_A | 790 oz | 300 oz | cancelled |
| `BR700004-3` | LP_B | 490 oz | 200 oz | cancelled |

Things to notice:
* One client order became **three LP trades** at three slightly different prices (4085.76 / 4085.73 / 4085.67).
  The client gets the average price, and later reconciliation (Lesson 07) has to match one client order to
  several LP trades.
* **Look at the quantities.** LP_A and LP_B receive gold in **ounces** (790, 490), but LP_C received **9.4**, the
  number of *lots*, not 940 ounces. That's a setup mistake in the bridge (incident **S10**, quantity mapping):
  LP_C was asked for 100 times less than intended. The bridge then believed LP_C filled 1.5 *lots* (150 oz) and sent
  only 790 oz to LP_A (940 − 150), while LP_C actually filled 1.5 *ounces*. Lessons 05 and 07 find this bug
  properly. Raw FIX shows it from the very first gold order of the day.
* The client asked for 9.4 lots and the platform recorded **6.5 lots** filled (1.5 + 3.0 + 2.0 lots). The other
  2.9 lots were never filled (the order was IOC, so nothing waits in the market).

---
## Cheat sheet

### Message types (tag `35`, MsgType)
| `35=` | Name | In plain words | Layer |
|---|---|---|---|
| `A` | Logon | "Hello, let's connect" | session |
| `0` | Heartbeat | "I'm still here" (sent after 10 s of silence) | session |
| `1` | TestRequest | "Are you there? Answer me with this ID" | session |
| `2` | ResendRequest | "I missed messages, send them again" | session |
| `3` | Reject | "Your message is broken (bad checksum, missing tag…)" | session |
| `4` | SequenceReset | "Skip these message numbers" (GapFill) | session |
| `5` | Logout | "Goodbye, closing the connection" | session |
| `V` | MarketDataRequest | "Please stream me prices for these symbols" | application |
| `W` | MarketDataSnapshotFullRefresh | "Here are my current prices" | application |
| `D` | NewOrderSingle | "Here is a new order" | application |
| `8` | ExecutionReport | "Here is what happened to your order" | application |

**Session layer** = messages that keep the connection itself healthy. **Application layer** = the business:
prices, orders, fills.

### Most common tags
| Tag | Name | Meaning / typical values |
|---|---|---|
| 8 | BeginString | FIX version, `FIX.4.4` |
| 9 | BodyLength | number of characters in the body |
| 10 | CheckSum | control number, 3 digits, always last |
| 34 | MsgSeqNum | message number, +1 each message, separate count in each direction |
| 49 / 56 | SenderCompID / TargetCompID | who sends / who receives |
| 52 | SendingTime | when sent, UTC, sender's clock |
| 43 | PossDupFlag | `Y` = "this message may be a copy of one I already sent" |
| 122 | OrigSendingTime | when the original was sent (on copies) |
| 108 | HeartBtInt | seconds between heartbeats |
| 141 | ResetSeqNumFlag | `Y` = restart message numbers at 1 |
| 112 | TestReqID | ID that must be repeated in the answer to a TestRequest |
| 7 / 16 | BeginSeqNo / EndSeqNo | range of messages asked for in a ResendRequest (`16=0` = "up to the latest") |
| 123 / 36 | GapFillFlag / NewSeqNo | in a SequenceReset: `123=Y` "skip", `36` the next number to expect |
| 11 | ClOrdID | **bridge's** order reference (`BR<client order>-<attempt>`) |
| 37 | OrderID | **LP's** order reference |
| 17 | ExecID | **LP's** reference for this one event (fill, reject, cancel). Unique: the key for matching trades |
| 55 | Symbol | EURUSD, GBPUSD, XAUUSD |
| 54 | Side | `1` Buy, `2` Sell |
| 38 | OrderQty | quantity ordered |
| 40 | OrdType | `1` Market, `2` Limit |
| 44 | Price | limit price |
| 59 | TimeInForce | `3` IOC (now or cancel), `1` GTC (until cancelled) |
| 150 | ExecType | what just happened: `0` New, `F` Trade (fill), `4` Canceled, `8` Rejected |
| 39 | OrdStatus | status of the whole order now: `0` New, `1` Partially filled, `2` Filled, `4` Canceled, `8` Rejected |
| 31 / 32 | LastPx / LastQty | price / quantity of **this** fill |
| 14 / 151 | CumQty / LeavesQty | total filled so far / still open |
| 6 | AvgPx | average fill price so far |
| 58 | Text | free text, usually the reason for a reject or logout |
| 262 | MDReqID | ID of the price subscription |
| 268 | NoMDEntries | how many price entries follow |
| 269 / 270 / 271 / 290 | MDEntryType / Px / Size / PriceLevel | `0` bid or `1` offer / price / size / level |

---
## Worked examples: 10 real messages, explained

Every message below is copied from the data files. For each one: the raw line, the important fields, and the
story. Tags are explained again every time, so you can read any example on its own.

### Example 1: the LP answers our Logon
```
20251015-11:29:58.507 IN  8=FIX.4.4|9=73|35=A|49=LP_A|56=BRIDGE01|34=1|52=20251015-11:29:58.504|98=0|108=10|141=Y|10=120|
```
| Field | Meaning |
|---|---|
| `IN` | we **received** this |
| `35=A` | MsgType = **Logon** |
| `49=LP_A` → `56=BRIDGE01` | from LP_A (SenderCompID) to our bridge (TargetCompID) |
| `34=1` | MsgSeqNum: LP_A's message number 1 to us today |
| `52=…11:29:58.504` | SendingTime on LP_A's clock |
| `108=10` | HeartBtInt: LP_A agrees to heartbeat every 10 seconds |
| `141=Y` | ResetSeqNumFlag: LP_A agrees to start numbering at 1 |

**Story:** we sent our Logon at 11:29:58.500 and LP_A's answer was logged at .507, so the handshake took **7 ms**.
The connection is now open. If an LP doesn't answer a Logon, the bridge shows the session as "disconnected"
and its prices are not available.

### Example 2: asking for prices
```
20251015-11:29:58.510 OUT 8=FIX.4.4|9=121|35=V|49=BRIDGE01|56=LP_A|34=2|52=20251015-11:29:58.510|262=MD-LP_A|263=1|264=2|265=0|146=3|55=EURUSD|55=GBPUSD|55=XAUUSD|10=223|
```
| Field | Meaning |
|---|---|
| `OUT` | we **sent** this |
| `35=V` | MsgType = **MarketDataRequest**: "please send me prices" |
| `34=2` | MsgSeqNum: our 2nd message to LP_A (the Logon was 1) |
| `262=MD-LP_A` | MDReqID: our name for this subscription |
| `263=1` | SubscriptionRequestType: `1` = "subscribe: keep sending updates" (`0` would mean one snapshot only) |
| `264=2` | MarketDepth: "send me **2 levels**" (best price and second best) |
| `265=0` | MDUpdateType: `0` = send the **full** price list every time |
| `146=3` | NoRelatedSym: 3 symbols follow (a repeating group again) |
| `55=EURUSD`, `55=GBPUSD`, `55=XAUUSD` | Symbol, three times: the 3 symbols we want |

**Story:** right after logging on, the bridge subscribes to 2-level prices for 3 symbols. From now on LP_A will
send `35=W` messages every time its price changes.

### Example 3: a price update
```
20251015-11:30:00.081 IN  8=FIX.4.4|9=220|35=W|49=LP_A|56=BRIDGE01|34=2|52=20251015-11:30:00.078|262=MD-LP_A-XAUUSD|55=XAUUSD|268=4|269=0|270=4085.32|271=600|290=1|269=0|270=4085.29|271=1200|290=2|269=1|270=4085.52|271=600|290=1|269=1|270=4085.55|271=1200|290=2|10=070|
```
| Field | Meaning |
|---|---|
| `35=W` | MsgType = **MarketDataSnapshotFullRefresh**: LP_A's full current prices |
| `55=XAUUSD` | Symbol: gold |
| `268=4` | NoMDEntries: 4 entries follow |
| `269=0\|270=4085.32\|271=600\|290=1` | entry 1: **bid** (`269=0`) at 4085.32 for 600 oz, level 1 (best) |
| `269=0\|270=4085.29\|271=1200\|290=2` | entry 2: bid at 4085.29 for 1,200 oz, level 2 |
| `269=1\|270=4085.52\|271=600\|290=1` | entry 3: **offer** (`269=1`) at 4085.52 for 600 oz, level 1 |
| `269=1\|270=4085.55\|271=1200\|290=2` | entry 4: offer at 4085.55 for 1,200 oz, level 2 |

**Story:** "I buy gold at 4085.32, I sell at 4085.52." Spread = 4085.52 − 4085.32 = 0.20 USD = **2 pips** (for
gold in this lab, 1 pip = 0.10 USD). Sent at .078 on LP_A's clock, logged by us at .081: it took about 3 ms to
arrive.

### Example 4: a heartbeat
```
20251015-11:30:08.510 OUT 8=FIX.4.4|9=55|35=0|49=BRIDGE01|56=LP_A|34=3|52=20251015-11:30:08.510|10=021|
```
| Field | Meaning |
|---|---|
| `35=0` | MsgType = **Heartbeat**: "I'm alive" |
| `34=3` | MsgSeqNum: our 3rd message to LP_A |
| (no body) | a heartbeat carries nothing else |

**Story:** our last message to LP_A was the price request at 11:29:58.510. Exactly **10 seconds** later (the
HeartBtInt agreed at Logon, tag `108=10`) we had sent nothing else, so the bridge sent a heartbeat to show the
connection is alive. Prices flow from LP_A to us, not the other way, so our side of the connection is often quiet.

### Example 5: an order and a reject, then a re-route and a fill
```
11:57:09.410 OUT ...35=D|49=BRIDGE01|56=LP_A|34=209|...|11=BR700335-1|1=BRIDGE01|55=EURUSD|54=1|...|38=218000|40=2|44=1.16280|59=3|10=190|
11:57:09.417 IN  ...35=8|49=LP_A|56=BRIDGE01|...|37=LP_A-O7003351|11=BR700335-1|17=LP_A-X7003351|150=8|39=8|55=EURUSD|54=1|38=218000|14=0|151=0|6=0|58=Internal error|...
11:57:09.418 OUT ...35=D|49=BRIDGE01|56=LP_B|34=183|...|11=BR700335-2|...|55=EURUSD|54=1|...|38=218000|40=2|44=1.16280|59=3|...
11:57:09.451 IN  ...35=8|49=LP_B|56=BRIDGE01|...|37=LP_B-O7003352|11=BR700335-2|17=LP_B-E000030|150=F|39=2|...|31=1.16280|32=218000|14=218000|151=0|6=1.16280|...
```
| Message | Key fields | Meaning |
|---|---|---|
| 1 (OUT) | `35=D` NewOrderSingle, `11=BR700335-1` ClOrdID, `54=1` Side Buy, `38=218000` OrderQty, `40=2` OrdType Limit, `44=1.16280` Price, `59=3` TimeInForce IOC | "LP_A, buy 218,000 EUR at 1.16280 or better, now or cancel" |
| 2 (IN) | `35=8` ExecutionReport, `150=8` ExecType **Rejected**, `39=8` OrdStatus Rejected, `14=0` CumQty (nothing filled), `58=Internal error` Text | "No." The reason is a problem on LP_A's side |
| 3 (OUT) | same order, `11=BR700335-2` (attempt 2), now to LP_B | the bridge tries the next LP 1 ms later |
| 4 (IN) | `150=F` ExecType **Trade**, `39=2` OrdStatus **Filled**, `31=1.16280` LastPx, `32=218000` LastQty, `151=0` LeavesQty (nothing left), `17=LP_B-E000030` ExecID | "Done: all 218,000 bought at 1.16280. My trade reference is LP_B-E000030" |

**Story:** first LP said no, second LP filled everything 41 ms after the first attempt. On the platform, the
client (Standard group) was filled at 1.16288: the LP price **plus a 0.8 pip markup** (the broker's margin for
that client group).

### Example 6: a partial fill and the cancelled rest
```
11:30:34.546 IN  ...35=8|49=LP_A|...|11=BR700004-2|17=LP_A-E000001|150=F|39=1|55=XAUUSD|54=2|38=790|31=4085.73|32=300|14=300|151=490|6=4085.73|...
11:30:34.547 IN  ...35=8|49=LP_A|...|11=BR700004-2|17=LP_A-C7000042|150=4|39=4|55=XAUUSD|54=2|38=790|14=300|151=0|6=4085.73|58=IOC remainder cancelled|...
```
| Message | Key fields | Meaning |
|---|---|---|
| 1 | `150=F` ExecType Trade, `39=1` OrdStatus **Partially filled**, `38=790` OrderQty, `32=300` LastQty, `14=300` CumQty, `151=490` LeavesQty | "Filled 300 of your 790 oz at 4085.73; 490 oz still open" |
| 2 | `150=4` ExecType **Canceled**, `39=4` OrdStatus Canceled, `14=300` CumQty, `151=0` LeavesQty, `58=IOC remainder cancelled` | "The 490 left are cancelled, because your order was IOC (fill now or cancel)" |

**Story:** LP_A only had 300 oz at that price. The bridge immediately sent the remaining 490 oz to LP_B
(`BR700004-3`). Notice the two ExecIDs (tag `17`): `E…` for the fill and `C…` for the cancel. Every event gets
its own ExecID.

### Example 7: "Are you there?" (TestRequest)
```
20251015-13:00:12.000 OUT 8=FIX.4.4|9=73|35=1|49=BRIDGE01|56=LP_C|34=690|52=20251015-13:00:12.000|112=TEST-130012|10=222|
```
| Field | Meaning |
|---|---|
| `35=1` | MsgType = **TestRequest**: "are you there?" |
| `56=LP_C` | TargetCompID: to LP_C |
| `112=TEST-130012` | TestReqID: "answer me with a Heartbeat that contains this same ID" |

**Story:** LP_C had sent **nothing at all** since 13:00:00: no prices, no heartbeats. After 12 seconds (the
10-second heartbeat interval plus a 20% margin) the bridge asks "are you there?". A healthy LP would answer
immediately with `35=0|112=TEST-130012`. LP_C doesn't answer.

### Example 8: giving up (Logout)
```
20251015-13:00:22.000 OUT 8=FIX.4.4|9=106|35=5|49=BRIDGE01|56=LP_C|34=692|52=20251015-13:00:22.000|58=Heartbeat timeout: no response to TestRequest|10=128|
```
| Field | Meaning |
|---|---|
| `35=5` | MsgType = **Logout**: "I'm closing this connection" |
| `58=Heartbeat timeout: no response to TestRequest` | Text: the reason |

**Story:** 10 seconds after the TestRequest, still nothing from LP_C, so the bridge closes the session. LP_C's
prices must now be removed from the price feed, and any order sent to LP_C in the last 22 seconds has an
**unknown result**. At 13:02:30 the bridge logs on again with `35=A` (Logon) and `141=Y` (ResetSeqNumFlag:
"restart message numbers from 1"), so both sides start counting again from 1. Lesson 02 tells the full story.

### Example 9: "I missed some messages, send them again" (ResendRequest)
```
20251015-12:45:10.179 OUT 8=FIX.4.4|9=70|35=2|49=BRIDGE01|56=LP_B|34=508|52=20251015-12:45:10.179|7=14126|16=0|10=210|
```
| Field | Meaning |
|---|---|
| `35=2` | MsgType = **ResendRequest** |
| `7=14126` | BeginSeqNo: "send me again everything from **your** message number 14,126…" |
| `16=0` | EndSeqNo: `0` = "…up to your latest one" |

**Story:** the bridge was expecting LP_B's message number 14,129 (tag `34`, MsgSeqNum) but received 14,134, so
5 messages (14,129 to 14,133) were lost on the way. It asks LP_B to send them again. But look: it asks from
**14,126**, three numbers too early, so it will also receive again messages it already has. That's a bug, and it
causes Example 10.

### Example 10: a copy of an old fill (PossDup)
```
20251015-12:45:10.192 IN  8=FIX.4.4|9=245|35=8|49=LP_B|56=BRIDGE01|34=14128|43=Y|122=20251015-12:45:09.823|52=20251015-12:45:10.187|37=LP_B-O7009651|11=BR700965-1|17=LP_B-E000075|150=F|39=2|55=EURUSD|54=1|38=200000|31=1.16498|32=200000|...
```
| Field | Meaning |
|---|---|
| `35=8` | MsgType = ExecutionReport |
| `34=14128` | MsgSeqNum: LP_B's message number 14,128, a number we **already received** a moment earlier |
| `43=Y` | PossDupFlag = **Y**: "this may be a duplicate of a message I already sent you" |
| `122=…12:45:09.823` | OrigSendingTime: when the original was sent |
| `52=…12:45:10.187` | SendingTime: when this copy was sent |
| `17=LP_B-E000075` | ExecID: the LP's trade reference, **the same** as the original |
| `150=F`, `39=2`, `32=200000`, `31=1.16498` | Trade, Filled, 200,000 EURUSD at 1.16498 |

**Story:** because of the bad ResendRequest in Example 9, LP_B correctly re-sends a fill we already had, and marks
it `43=Y` ("possible duplicate"). The rule is simple: **before acting on a `43=Y` message, check whether you've
already seen that ExecID** (tag `17`). If yes, ignore it. Our bridge didn't, and the trading platform booked the
same 2-lot trade **twice** for client order 700965. Lesson 07 finds the duplicate deal.

---
**Next:** Lesson 02, the session layer: heartbeats, message numbers, gaps and disconnections, in detail.